# Compute GeoCLIP outputs for Zero-Shot city localisation

This is done to establish feasibility of using CLIP-based models for global geolocalisation.

In [ ]:
import torch
from geoclip import GeoCLIP
from haversine import haversine, Unit
from pathlib import Path
import os
import pickle
from glob import glob
from tqdm import tqdm
import pandas as pd

## Load model

In [ ]:
model = GeoCLIP()
model.to("cuda")

## Setup

In [ ]:
os.makedirs("./csv_files", exist_ok=True)

In [ ]:
choices = [
    "Chicago",
    "London",
    "Los Angeles",
    "Melbourne",
    "Miami",
    "New York",
    "San Francisco",
    "Singapore",
    "Sydney",
    "Toronto",
]

choices_coords = [
    (41.8832, -87.6324),  # Chicago
    (51.5072, -0.1276),  # London
    (35.0549, -118.2426),  # Los Angeles
    (-37.8136, 144.9631),  # Melbourne
    (25.7617, -80.1918),  # Miami
    (40.7128, -74.0060),  # New York
    (37.7749, -122.4194),  # San Francisco
    (1.3521, 103.8198),  # Singapore
    (-33.87287, 151.2057),  # Sydney
    (43.6532, -79.3832),  # Toronto
]


In [ ]:
def haversine_distance(loc1, loc2):
    distance = haversine(loc1, loc2)
    return distance

**Note**: Update this path accordingly

In [ ]:
base_dir = "path_to_filtered_instacities_dataset"  # Update this path accordingly

## Running GeoCLIP

In [ ]:
for city_name in os.listdir(base_dir):
    print(f"Processing {city_name}...")
    all_image_names = []
    all_majority_cities = []

    # Load images
    all_images = glob(f"{os.path.join(base_dir, city_name)}/*.jpg")

    # Compute and store embeddings
    for image_path in tqdm(all_images):
        score_key = Path(image_path).stem
        with torch.no_grad():
            top_pred_gps, top_pred_prob = model.predict(image_path, top_k=1)
            lat, lon = top_pred_gps[0]
            predicted_loc = (lat, lon)
            distances = [haversine_distance(predicted_loc, x) for x in choices_coords]
            distances = torch.Tensor(distances)
            class_index = int(
                distances.argmin().item()
            )  # As this is distances we select the one which is the closest

            all_image_names.append(score_key)
            all_majority_cities.append(choices[class_index])

    # Store scores
    df = pd.DataFrame(
        {"image_name": all_image_names, "majority_city": all_majority_cities}
    )

    df.to_csv(f"./csv_files/geoclip_{city_name}.pkl", index=False)
    torch.cuda.empty_cache()